## Student Attendance Analytics with Pandas

In [ ]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [ ]:
df = pd.read_excel("CSEB_Attendance.xlsx")
df

#### Section 1: Data Inspection

In [ ]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
#-----------------------------Write Solution Code-----------------------------
#the first 10 rows
df.head(10)
#the shape of the DataFrame
df.shape
#the data types of each column
df.dtypes

In [ ]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """
#-----------------------------Write Solution Code-----------------------------
#check for missing values in each column
df.isnull().sum()
#Identify which columns have null values (especially Last Present Date with "Never") and report the count.
(df['Last Present Date']=="Never").sum()



In [ ]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
#-----------------------------Write Solution Code-----------------------------
df.rename(columns={'Roll No': 'roll_no', 'Student Name': 'student_name', 'Attendance %': 'attendance_pct'}, inplace=True)
df

In [ ]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
#-----------------------------Write Solution Code-----------------------------

#### Section 2: Filtering & Selection

In [ ]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
#-----------------------------Write Solution Code-----------------------------
df_filtered = df[df['Status'] == 'Critical']
df_filtered = df_filtered.sort_values(by='attendance_pct', ascending=True)
df_filtered

In [ ]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
#-----------------------------Write Solution Code-----------------------------
df_absent = df[df['Present'] == 0]
df_absent[['roll_no', 'student_name', 'Consecutive Absences']]

In [ ]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """
#-----------------------------Write Solution Code-----------------------------
df_safe = df.loc[df['Status'] == 'Safe']
df_safe

df_first= df.iloc[0:15, 1:6]
df_first

In [ ]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
#-----------------------------Write Solution Code-----------------------------
df_consecutive_absent = df[df['Consecutive Absences'] >= 5]
df_consecutive_absent[['student_name', 'Status', 'Last Present Date']]

### Section 3: Grouping & Aggregation 

In [ ]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""
#-----------------------------Write Solution Code-----------------------------
df.groupby('Status').agg({'roll_no': 'count', 'attendance_pct': 'mean', 'Consecutive Absences': 'max'})

In [ ]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""
#-----------------------------Write Solution Code-----------------------------
df['Attendance category']=df['attendance_pct'].apply(lambda x: "Excellent" if int(str(x).replace('%',' ')) >= 90 else "Good" if int(str(x).replace('%',' ')) >= 75 else "Average" if int(str(x).replace('%',' ')) >= 60 else "Poor")
df.groupby('Attendance category').size()

In [ ]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
#-----------------------------Write Solution Code-----------------------------
df.agg({'Total Classes': 'sum', 'Present': 'sum', 'Absent': 'sum'})

In [ ]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""
#-----------------------------Write Solution Code-----------------------------
df.groupby('Status').agg({'Present': ['min', 'max', 'mean'], 'Absent': 'sum'})

### Section 4: Feature Engineering & Transformation

In [ ]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""
#-----------------------------Write Solution Code-----------------------------
df['Risk Score'] = (df['Absent'] * 0.4) + (df['Consecutive Absences'] * 0.6)
df.nlargest(10, 'Risk Score')

In [ ]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
#-----------------------------Write Solution Code-----------------------------
# df['Classes Needed (Numeric)'] = df['Classes Needed (75%)'].str.extract(r'(\d+)')
# df['Classes Needed (Numeric)'] = df['Classes Needed (Numeric)'].fillna(0).astype(int)
# df


In [ ]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """
#-----------------------------Write Solution Code-----------------------------
df['Attendance Flag'] = df['attendance_pct'].apply(lambda x: "Defaulter" if int(str(x).replace('%',' ')) < 75 else "Safe")
df

In [ ]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
#-----------------------------Write Solution Code-----------------------------
df['Attendance Range'] = pd.cut(df['attendance_pct'].apply(lambda x: int(str(x).replace('%',' '))), bins=[0, 50, 75, 90, 100], labels=["Critical", "Warning", "Good", "Excellent"])

### Section 5: Advanced Analysis 

In [ ]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
#-----------------------------Write Solution Code-----------------------------
pd.pivot_table(df, index='Status', columns='Attendance category', values='roll_no', aggfunc='count', fill_value=0)

In [ ]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
#-----------------------------Write Solution Code-----------------------------
df['Base Name'] = df['student_name'].str.upper().str.strip()
df[df.duplicated(subset='Base Name', keep=False)]


In [ ]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
#-----------------------------Write Solution Code-----------------------------


In [ ]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""
#-----------------------------Write Solution Code-----------------------------
df[df['Attendance Category'] == 'Critical'].to_csv('critical_students.csv', index=False)
df.groupby('Status').agg({'risk_score': 'mean'}).to_csv('summary_statistics.csv', index=True)
df.nlargest(10, 'risk_score').to_csv('top_10_students.csv', index=False)    